# Citation-need detection — training

Trains a sentence classifier: **does this sentence need a citation?**

The trick that makes this cheap: in a *published* open-access paper, the authors and
reviewers already decided. A sentence that carries a citation marker is a positive
example; strip the marker and you have the input. Sentences that never carried one, in
the same paper and section, are negatives.

This replaces the hand-written rules in `src/lib/citation/citationNeed.ts`.

**Scope note.** This model asks *"should something be cited here?"*. It never proposes a
source, never rewrites text, and is not trained against any detector. Fabricated citations
are impossible by construction — the output is a single probability.

Runtime: ~40 min on Kaggle T4. Settings → Accelerator **GPU T4 ×2**, Internet **On**.

In [ ]:
!pip -q install "transformers>=4.44" "datasets>=2.20" "accelerate>=0.33" \
    "scikit-learn" "optimum[onnxruntime]" "onnx" "onnxruntime"

import os, re, json, random
import numpy as np
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 1. Data

`unarxive_citrec` is already framed as citation recommendation, so it needs the least
wrangling. Swap in PMC OA or S2ORC by replacing `load_raw()` — everything downstream
expects a list of `{text, label, section}`.

**Domain choice.** unarXive is physics/CS/maths-heavy. For a biomedical paper, use the PMC
OA loader instead — a citation-need model transfers poorly across fields because what counts
as common knowledge differs.

In [ ]:
from datasets import load_dataset

# Citation markers to strip so the model can't cheat by spotting the bracket
CITE_PATTERNS = [
    r'\{\{cite:[^}]*\}\}',              # unarXive placeholder
    r'\[\d+(?:\s*[,\u2013-]\s*\d+)*\]',  # IEEE [1], [2-4]
    r'\([A-Z][A-Za-z\'-]+(?: et al\.)?,?\s*\d{4}[a-z]?\)',  # APA (Smith, 2020)
    r'\{\{formula:[^}]*\}\}',
]
CITE_RE = re.compile('|'.join(CITE_PATTERNS))

def strip_citations(text: str) -> str:
    return re.sub(r'\s+', ' ', CITE_RE.sub('', text)).strip()


def load_raw(max_rows=250_000):
    """Yield {text, label, section}. label=1 means the sentence carried a citation."""
    ds = load_dataset('saier/unarxive_citrec', split='train', streaming=True)
    out = []
    for row in ds:
        text = row.get('text') or ''
        if not text:
            continue
        label = 1 if CITE_RE.search(text) else 0
        clean = strip_citations(text)
        # Very short or very long fragments are usually parser noise
        wc = len(clean.split())
        if wc < 8 or wc > 120:
            continue
        out.append({'text': clean, 'label': label, 'section': row.get('section', '') or ''})
        if len(out) >= max_rows:
            break
    return out

raw = load_raw()
print('rows:', len(raw), '| positive rate:', np.mean([r['label'] for r in raw]).round(3))

### Leakage guards

Two failure modes that would inflate the score and produce a useless model:

1. **Marker residue** — any leftover bracket makes the task trivial. Assert none survive.
2. **Class imbalance** — cited sentences are the minority; balance so precision means
   something.

Split by *paper*, not by sentence, when your loader exposes a paper id — sentences from one
paper share vocabulary and would leak across the split.

In [ ]:
leaked = [r for r in raw if CITE_RE.search(r['text'])]
assert not leaked, f'{len(leaked)} rows still contain citation markers — model would cheat'

pos = [r for r in raw if r['label'] == 1]
neg = [r for r in raw if r['label'] == 0]
k = min(len(pos), len(neg))
random.shuffle(pos); random.shuffle(neg)
balanced = pos[:k] + neg[:k]
random.shuffle(balanced)
print(f'balanced: {len(balanced)} ({k} per class)')

n = len(balanced)
train_rows = balanced[: int(0.8 * n)]
val_rows   = balanced[int(0.8 * n) : int(0.9 * n)]
test_rows  = balanced[int(0.9 * n) :]
print(len(train_rows), len(val_rows), len(test_rows))

## 2. Baseline first

Always get a cheap number before spending GPU time. If TF-IDF + logistic regression is
close to the transformer, ship the baseline — it's 200 KB instead of 40 MB and runs
instantly in the browser.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, precision_recall_fscore_support

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=200_000, sublinear_tf=True)
Xtr = vec.fit_transform([r['text'] for r in train_rows])
Xte = vec.transform([r['text'] for r in test_rows])
ytr = [r['label'] for r in train_rows]
yte = [r['label'] for r in test_rows]

lr = LogisticRegression(max_iter=2000, C=1.0)
lr.fit(Xtr, ytr)
print(classification_report(yte, lr.predict(Xte), digits=3))

## 3. Fine-tune SciBERT

`allenai/scibert_scivocab_uncased` has a scientific vocabulary, which matters a lot here —
general-domain BERT wastes tokens on subword-splitting terms like *aptasensor*.

For a smaller browser payload use `microsoft/deberta-v3-small` instead; it quantizes to
~40 MB versus SciBERT's ~110 MB.

In [ ]:
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from datasets import Dataset

MODEL_NAME = 'allenai/scibert_scivocab_uncased'   # or 'microsoft/deberta-v3-small'
MAX_LEN = 128

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

def encode(rows):
    d = Dataset.from_list(rows)
    return d.map(lambda b: tok(b['text'], truncation=True, max_length=MAX_LEN),
                 batched=True, remove_columns=['text', 'section'])

ds_train, ds_val, ds_test = encode(train_rows), encode(val_rows), encode(test_rows)

def metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
    # F0.5 weights precision 4x — a wrong "cite this" costs more than a miss (plan.md §8)
    f05 = (1.25 * p * r / (0.25 * p + r)) if (p + r) else 0.0
    return {'precision': p, 'recall': r, 'f1': f1, 'f0.5': f05}

args = TrainingArguments(
    output_dir='out',
    num_train_epochs=2,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=128,
    learning_rate=2e-5,
    warmup_ratio=0.06,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f0.5',
    logging_steps=100,
    report_to='none',
    seed=SEED,
)

trainer = Trainer(model=model, args=args, train_dataset=ds_train, eval_dataset=ds_val,
                  compute_metrics=metrics, data_collator=DataCollatorWithPadding(tok))
trainer.train()
print(trainer.evaluate(ds_test))

## 4. Pick the shipping threshold on validation

Do **not** report the best threshold found on test — that number won't survive contact with
real papers. Choose on validation, then report what it scores on test. Ship the threshold
alongside the weights.

In [ ]:
import torch.nn.functional as F

def probs(ds):
    logits = torch.tensor(trainer.predict(ds).predictions)
    return F.softmax(logits, dim=-1)[:, 1].numpy()

val_p, val_y = probs(ds_val), np.array([r['label'] for r in val_rows])

TARGET_PRECISION = 0.85
best = None
for t in np.arange(0.30, 0.96, 0.01):
    pred = (val_p >= t).astype(int)
    p, r, _, _ = precision_recall_fscore_support(val_y, pred, average='binary', zero_division=0)
    if p >= TARGET_PRECISION and (best is None or r > best[2]):
        best = (float(t), float(p), float(r))

THRESHOLD = best[0] if best else 0.5
print('chosen on validation:', best)

test_p, test_y = probs(ds_test), np.array([r['label'] for r in test_rows])
p, r, _, _ = precision_recall_fscore_support(test_y, (test_p >= THRESHOLD).astype(int),
                                             average='binary', zero_division=0)
print(f'TEST @ {THRESHOLD:.2f} -> precision {p:.3f}  recall {r:.3f}')
print('Ship only if precision >= 0.85 (plan.md §8).')

## 5. Sanity-check on real sentences

Metrics hide the failure that matters: flagging the authors' own results. Check by hand
before shipping — these are the same cases asserted in `citationNeed.test.ts`.

In [ ]:
probe = [
    ('Several previous studies have shown that graphene biosensors achieve femtomolar sensitivity.', 'should be HIGH'),
    ('It has been widely reported that AlGaN/GaN HEMTs exhibit high electron mobility.',            'should be HIGH'),
    ('In this work we simulated the device using Silvaco ATLAS.',                                   'should be LOW'),
    ('Our results show a subthreshold swing of 122 mV per decade.',                                 'should be LOW'),
    ('As shown in Figure 4, the threshold voltage shifts with charge density.',                     'should be LOW'),
    ('The simulation was repeated for each bias condition.',                                        'should be LOW'),
]

model.eval()
with torch.no_grad():
    enc = tok([t for t, _ in probe], return_tensors='pt', padding=True,
              truncation=True, max_length=MAX_LEN).to(model.device)
    p1 = F.softmax(model(**enc).logits, dim=-1)[:, 1].cpu().numpy()

for (text, expect), score in zip(probe, p1):
    flag = 'FLAG ' if score >= THRESHOLD else '  -  '
    print(f'{flag} {score:.3f}  ({expect:<14}) {text[:72]}')

## 6. Export quantized ONNX for the browser

The app parses in-browser so the paper never leaves the machine (`plan.md` §2) — the model
has to follow it there. Quantized ONNX + `transformers.js` keeps that property.

In [ ]:
from optimum.onnxruntime import ORTModelForSequenceClassification, ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig

OUT = 'citation_need_onnx'
trainer.save_model('best'); tok.save_pretrained('best')

ort = ORTModelForSequenceClassification.from_pretrained('best', export=True)
ort.save_pretrained(OUT); tok.save_pretrained(OUT)

qconfig = AutoQuantizationConfig.avx512_vnni(is_static=False, per_channel=False)
ORTQuantizer.from_pretrained(OUT).quantize(save_dir=OUT, quantization_config=qconfig)

# Ship the threshold with the weights — the app must not guess it
json.dump({'threshold': float(THRESHOLD), 'base_model': MODEL_NAME,
           'test_precision': float(p), 'test_recall': float(r)},
          open(f'{OUT}/inference_config.json', 'w'), indent=2)

for f in sorted(os.listdir(OUT)):
    print(f, round(os.path.getsize(f'{OUT}/{f}') / 1e6, 1), 'MB')

## 7. Wiring it back into the app

1. Copy `citation_need_onnx/` → `public/models/citation-need/`
2. `npm install @huggingface/transformers`
3. Lazy-load behind a "deep citation check" toggle — don't pay 40 MB on first paint
4. Keep `citationNeed.ts` as the fallback while the model loads

```ts
import { pipeline, env } from '@huggingface/transformers';
env.allowRemoteModels = false;          // keep everything local
env.localModelPath = '/models/';

const clf = await pipeline('text-classification', 'citation-need');
const { threshold } = await fetch('/models/citation-need/inference_config.json').then(r => r.json());

for (const s of paper.sentences) {
  const [{ label, score }] = await clf(s.text);
  if (label === 'LABEL_1' && score >= threshold) { /* emit citation_need finding */ }
}
```

Batch the sentences and run it in a Web Worker — a 6,000-word paper is ~300 sentences and
will block the main thread otherwise.